In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from Bio import SeqIO
from tqdm import tqdm
import json
import os
import logging
import numpy as np
import matplotlib.patches as patches
import random
from scipy.stats import mannwhitneyu

# Import all our custom pipeline modules
from instanexus import preprocessing
from instanexus import assembly
from instanexus import visualization
from instanexus import helpers

# Set up logging to see the pipeline's progress
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
pd.options.display.max_colwidth = None

In [ ]:
os.chdir('../../../../')

print(f"Current working directory: {os.getcwd()}")

In [ ]:
FIGURES_DIR = Path("figures")
print(FIGURES_DIR)

In [ ]:
# Path to the new raw data you want to test
INPUT_CSV = "inputs/nb6.csv"


# Base folder for all results
BASE_OUTPUT_FOLDER = "outputs_notebook"

# Paths to your static database files$
METADATA_PATH = "json/sample_metadata.json"
CONTAMINANTS_PATH = "fasta/contaminants.fasta"

# --- 2. Define Pipeline Parameters ---
RUN_NAME = Path(INPUT_CSV).stem
REFERENCE_MODE = True
CHAIN = ""

# Filtering params
CONFIDENCE_THRESHOLD = 0.8
#MASS_ERR_LIMIT = 20
MIN_LENGTH = 7
#MAX_IRT_ERROR = 60
#MIN_ENTROPY = 1
#PROSIT_FILTER = True
FDR_THRESHOLD = 0.1
#Z_SCORE_THRESHOLD = -0.5

# Assembly params
ASSEMBLY_MODE = "dbg_weighted"
KMER_SIZE = 6
MIN_OVERLAP = 3
SIZE_THRESHOLD = 10

MIN_IDENTITY = 1
MAX_MISMATCHES = 0

# Clustering params
MIN_SEQ_ID = 0.85
COVERAGE = 0.8

In [ ]:
base_output_folder = Path(BASE_OUTPUT_FOLDER) / RUN_NAME

# Build the unique experiment folder name
folder_name_parts = [f"{ASSEMBLY_MODE}"]

if CONFIDENCE_THRESHOLD is not None:
    folder_name_parts.append(f"c{CONFIDENCE_THRESHOLD}")

if "dbg" in ASSEMBLY_MODE:
    folder_name_parts.append(f"ks{KMER_SIZE}")

folder_name_parts.append(f"mo{MIN_OVERLAP}")
folder_name_parts.append(f"ts{SIZE_THRESHOLD}")

if REFERENCE_MODE:
    folder_name_parts.extend([f"mi{MIN_IDENTITY}", f"mm{MAX_MISMATCHES}"])

run_folder_name = "_".join(folder_name_parts)
experiment_folder = base_output_folder / run_folder_name


# --- Define a Run ID for logging ---
run_id_str = f"[{RUN_NAME} @ {run_folder_name}]"

logger.info(f"Pipeline starting for run: {run_id_str}")
logger.info(f"All results will be saved to: {experiment_folder}")

In [ ]:
sample_metadata = preprocessing.get_sample_metadata(
    run=RUN_NAME, 
    chain=CHAIN, 
    json_path=METADATA_PATH
)

In [ ]:
proteases = sample_metadata["proteases"]
protein = sample_metadata["protein"]
protein_norm = preprocessing.normalize_sequence(protein)

In [ ]:
print(f"Sample uses proteases: {proteases}")
print(f"Protein sequence length: {len(protein)} amino acids")
print(f"Normalized protein sequence: {protein_norm}")

In [ ]:
original_data = pd.read_csv(INPUT_CSV)

In [ ]:
original_data.columns

In [ ]:
cols_to_keep = [
    'experiment_name',
    'prediction_untokenised',
    'instanovo_token_log_probabilities',
    'calibrated_confidence',    
    'psm_q_value',
    'delta_mass_ppm',
    'Mass Error',               
    'is_missing_prosit_features', 
    'ion_match_intensity',
    'ion_matches',
    'iRT',
    'iRT error',
    'is_missing_irt_error',
    'predicted iRT',
    'margin',
    'entropy',
    'z-score'
    ]

data = original_data[cols_to_keep].copy()

In [ ]:
data.rename(columns={'calibrated_confidence': 'conf'}, inplace=True)

In [ ]:
data["protease"] = data["experiment_name"].apply(
    lambda name: preprocessing.extract_protease(name, proteases)
)

protease_col = data.pop("protease")
data.insert(data.columns.get_loc("prediction_untokenised") + 1, "protease", protease_col)

In [ ]:
data = data.dropna(subset=["prediction_untokenised"])

In [ ]:
data["cleaned_preds"] = data["prediction_untokenised"].apply(preprocessing.remove_modifications)

# move cleaned_preds next to prediction_untokenised
cleaned_preds_col = data.pop("cleaned_preds")
data.insert(data.columns.get_loc("prediction_untokenised") + 1, "cleaned_preds", cleaned_preds_col)

In [ ]:
cleaned_psms = data["cleaned_preds"].tolist()

In [ ]:
filtered_psms = preprocessing.filter_contaminants(
    cleaned_psms, RUN_NAME , CONTAMINANTS_PATH
)

In [ ]:
data = data[data["cleaned_preds"].isin(filtered_psms)]

In [ ]:
data.drop(columns=['prediction_untokenised'], inplace=True)

In [ ]:
data["mapped"] = data["cleaned_preds"].apply(
    lambda x: "True" if x in protein_norm else "False"
)

In [ ]:
data = data[data['cleaned_preds'].str.len() >= MIN_LENGTH]

In [ ]:
# show me value counts of mapped vs unmapped
data['mapped'].value_counts()

In [ ]:
def add_quantification_data(df_main, run_name, fdr_threshold, inputs_folder="inputs"):
    """
    Filters df_main by FDR, then looks for a quantification file ({run_name}_quant_scores.csv).
    Merges the abundance data into the filtered dataframe.
    """
    if fdr_threshold is not None:
        if "psm_q_value" in df_main.columns:
            initial_len = len(df_main)
            df_main = df_main[df_main['psm_q_value'] <= fdr_threshold].copy()
            logger.info(f"FDR Filter applied inside merge function: {initial_len} -> {len(df_main)} rows (<= {fdr_threshold})")
        else:
            logger.warning("FDR threshold provided but 'psm_q_value' column missing. Skipping filter.")

    quant_file_name = f"{run_name}_quant_scores.csv"
    quant_file_path = Path(inputs_folder) / quant_file_name
    
    if not quant_file_path.exists():
        logger.warning(f"Quantification file NOT FOUND: {quant_file_path}")
        logger.warning("Skipping abundance merging. 'peptide_abundance' will be missing.")
        return df_main

    logger.info(f"Found quantification file: {quant_file_path}")
    
    try:
        df_quant = pd.read_csv(quant_file_path)
        
        if "cleaned_preds" not in df_quant.columns or "total_abundance_norm" not in df_quant.columns:
            logger.warning(f"Quantification file format error. Missing columns in {quant_file_path}")
            return df_main

        df_quant_summed = df_quant.groupby('cleaned_preds', as_index=False)['total_abundance_norm'].sum()  
        df_quant_summed.rename(columns={'total_abundance_norm': 'peptide_abundance'}, inplace=True) 
        df_merged = pd.merge(df_main, df_quant_summed, on='cleaned_preds', how='left')
        df_merged['peptide_abundance'] = df_merged['peptide_abundance'].fillna(0)
        
        logger.info(f"Quantification data merged successfully. Output rows: {len(df_merged)}")
        return df_merged

    except Exception as e:
        logger.error(f"Error merging quantification data: {e}")
        return df_main

In [ ]:
data_abundance = add_quantification_data(data, RUN_NAME, FDR_THRESHOLD)

In [ ]:
sequences = data_abundance['cleaned_preds'].tolist()

In [ ]:
assembler = assembly.Assembler(
    mode="dbg_weighted",
    kmer_size=7,
    min_overlap=3,
    size_threshold=10,
    min_weight=2
)

In [ ]:
scaffolds = assembler.run(sequences=sequences, df_full=data_abundance)

In [ ]:
mapped_scaffolds = visualization.process_protein_contigs_scaffold(
    scaffolds, protein_norm, 10, 0.8)

In [ ]:
def mapping_substitutions_seaborn(
    mapped_sequences,
    prot_seq,
    bar_colors=None,
    output_folder=".",
    output_file=None,
    contig_color="#1f78b4",
    show_figure=False,
):
    sns.set_style("white")
    sns.set_context("paper", font_scale=1.2)

    default_colors = {
        "match": contig_color,
        "mismatch": "#b30000",
        "D_to_N": "#000000",
        "E_to_Q": "#A8A29E",
    }
    colors = {**default_colors, **(bar_colors or {})}

    common_height = 0.25
    ref_bar_height = common_height
    pep_bar_height = common_height
    track_spacing = 0.30
    base_y_offset = 0.4

    fig, ax = plt.subplots(figsize=(15, 5)) 

    ref_rect = patches.Rectangle(
        (0, 0), len(prot_seq), ref_bar_height,
        linewidth=0, 
        edgecolor='none', 
        facecolor='#e6f0ef',
        zorder=0
    )
    ax.add_patch(ref_rect)

    tracks = {}

    for seq, mapping in tqdm(mapped_sequences, desc="Plotting substitutions"):
        start_index, end_index, mismatches, _ = mapping
        
        placed = False
        sorted_tracks = sorted(tracks.keys())
        
        current_track_num = 0
        
        for track_num in sorted_tracks:
            track_segments = tracks[track_num]
            if not any(max(s, start_index) < min(e, end_index) for s, e in track_segments):
                tracks[track_num].append((start_index, end_index))
                current_track_num = track_num
                placed = True
                break
        
        if not placed:
            current_track_num = len(tracks)
            tracks[current_track_num] = [(start_index, end_index)]

        current_y = base_y_offset + (current_track_num * track_spacing)
        
        base_rect = patches.Rectangle(
            (start_index, current_y), 
            end_index - start_index, 
            pep_bar_height,
            linewidth=0,
            edgecolor='none',
            facecolor=colors["match"],
            alpha=1.0
        )
        ax.add_patch(base_rect)

        for mismatch in mismatches:
            abs_index = start_index + mismatch
            
            if abs_index >= len(prot_seq) or mismatch >= len(seq):
                continue

            ref_aa = prot_seq[abs_index]
            query_aa = seq[mismatch]

            if query_aa == "D" and ref_aa == "N":
                mut_color = colors["D_to_N"]
            elif query_aa == "E" and ref_aa == "Q":
                mut_color = colors["E_to_Q"]
            else:
                mut_color = colors["mismatch"]

            mut_rect = patches.Rectangle(
                (abs_index, current_y), 
                1, 
                pep_bar_height,
                linewidth=0,
                edgecolor='none',
                facecolor=mut_color,
                zorder=10 
            )
            ax.add_patch(mut_rect)

    max_track = len(tracks) if tracks else 0
    ylim_top = base_y_offset + (max_track * track_spacing) + 0.5
    
    ax.set_xlim(0, len(prot_seq))
    ax.set_ylim(0, ylim_top)
    
    ax.set_xlabel("Residue Position", fontsize=14)
    ax.set_yticks([])
    
    sns.despine(left=True, bottom=True)
    ax.spines['bottom'].set_visible(False)
    ax.tick_params(axis='x', which='both', bottom=True, top=False, labelbottom=True)

    legend_patches = [
        patches.Patch(color=colors["match"], label="Match"),
        patches.Patch(color=colors["mismatch"], label="Mismatch"),
        patches.Patch(color=colors["D_to_N"], label="Seq:D → Ref:N"),
        patches.Patch(color=colors["E_to_Q"], label="Seq:E → Ref:Q"),
    ]
    
    ax.legend(
        handles=legend_patches, 
        title="",
        loc='upper center', 
        bbox_to_anchor=(0.5, 1.05),
        ncol=4,
        frameon=False,
        fontsize=11
    )

    plt.tight_layout()

    if output_file:
        os.makedirs(output_folder, exist_ok=True)
        full_path = os.path.join(output_folder, output_file)
        plt.savefig(full_path, format='svg', dpi=300, bbox_inches='tight')
        print(f"Figure saved to {full_path}")

    if show_figure:
        plt.show()
    else:
        plt.close()

In [ ]:
mapping_substitutions_seaborn(
    mapped_scaffolds,
    protein_norm,
    output_folder=FIGURES_DIR,
    output_file=f"fig4d_{RUN_NAME}_substitutions_mapping.svg",
    show_figure=True
)

In [ ]:
def plot_coverage_boxplot_seaborn_layered(
    file_path: str, 
    output_image: str = f'{FIGURES_DIR}/fig3a_coverage_boxplot_layered.svg'
):

    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"File not found: {file_path}. Generating dummy data.")
        data = {
            'assembly_method': ['greedy (Contigs)']*15 + ['greedy (Scaffolds)']*15,
            'coverage': np.concatenate([
                np.random.normal(0.85, 0.08, 15), 
                np.random.normal(0.92, 0.05, 15)
            ])
        }
        df = pd.DataFrame(data)

    df['Type'] = df['assembly_method'].apply(
        lambda x: 'Contigs' if 'Contigs' in x else 'Scaffolds'
    )
    
    colors = {"Contigs": "#a6cee3", "Scaffolds": "#1f78b4"}

    fig_w, fig_h = (3.5, 4) 

    plt.figure(figsize=(fig_w, fig_h))

    ax = sns.boxplot(
        data=df, 
        x='Type', 
        y='coverage', 
        palette=colors,
        width=0.5,
        showfliers=False, 
        linewidth=1,
    )
    
    sns.stripplot(
        data=df, 
        x='Type', 
        y='coverage', 
        palette=colors,     
        size=6,
        jitter=0.15,
        edgecolor="#333333",
        linewidth=0.8,      
        alpha=0.9,          
        ax=ax               
    )
    ax.set_ylabel('Coverage', fontsize=10)
    ax.set_xlabel('', fontsize=10)
    
    ax.tick_params(axis='both', which='major', labelsize=9)
    
    sns.despine()

    os.makedirs(os.path.dirname(output_image) if os.path.dirname(output_image) else '.', exist_ok=True)
    plt.savefig(output_image, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
plot_coverage_boxplot_seaborn_layered('outputs/_summary_tables/dbg_weighted/best_results_Nanobodies_dbg_weighted.csv')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# Assicurati di avere set_publication_style definita/importata

def plot_psm_depth_standardized(
    reference_seq: str, 
    peptides: list, 
    cdrs: dict, 
    output_file: str = 'fig_4C_matplotlib.svg'
):
    """
    Standardized PSM Depth Plot.
    Target Size: 2/3 of A4 Page Width (~4.75 inches).
    """
        
    # --- Helper: Robust Search (L/I Tolerant) ---
    def chars_equal(a, b):
        return (a in ['L', 'I'] and b in ['L', 'I']) or a == b

    def find_all_occurrences(ref, pep):
        starts = []
        for pos in range(len(ref) - len(pep) + 1):
            if all(chars_equal(ref[pos+i], pep[i]) for i in range(len(pep))):
                starts.append(pos)
        return starts

    # 2. Calcolo Depth
    depth = np.zeros(len(reference_seq), dtype=int)
    for pep in peptides:
        found_indices = find_all_occurrences(reference_seq, pep)
        for start_idx in found_indices:
            end_idx = start_idx + len(pep)
            depth[start_idx:end_idx] += 1
            
    # 3. Dimensioni Fisiche (2/3 Page Width)
    FULL_A4_WIDTH = 7.1
    FIG_W = FULL_A4_WIDTH * (2/3)  # ~4.73 pollici
    FIG_H = 3.0                    # Altezza standard
    
    # layout='constrained' è cruciale per gestire i margini in spazi più stretti
    fig = plt.figure(figsize=(FIG_W, FIG_H), layout='constrained')
    ax = fig.gca()
    
    # 4. Plotting
    x = np.arange(len(reference_seq))
    
    ax.plot(x, depth, color='steelblue', linewidth=1.5, label='PSM Depth')
    ax.fill_between(x, depth, color='steelblue', alpha=0.2)
    
    # 5. CDR Highlights
    highlight_colors = {
        "CDR1": "orange", 
        "CDR2": "lightgreen", 
        "CDR3": "deepskyblue"
    }
    
    max_y = depth.max() * 1.1 if depth.max() > 0 else 1.0
    
    for label, (start, end) in cdrs.items():
        # start-1 per correzione 0-based
        ax.axvspan(start-1, end, color=highlight_colors.get(label, 'gray'), alpha=0.3, zorder=0)
        
        # Etichetta un po' più piccola (fontsize=8) per stare nel grafico più stretto
        ax.text((start-1 + end)/2, max_y, label, 
                ha='center', va='top', fontsize=8, fontweight='bold', color='black')

    # 6. Formatting
    ax.set_title('PSM depth across protein sequence', fontsize=10)
    ax.set_xlabel('Amino acid position', fontsize=9)
    ax.set_ylabel('Depth (PSMs)', fontsize=9)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    ax.set_xlim(0, len(reference_seq))
    ax.set_ylim(0, max_y * 1.05)
    
    # 7. Save
    os.makedirs(os.path.dirname(output_file) if os.path.dirname(output_file) else '.', exist_ok=True)
    
    # Manteniamo bbox_inches=None per rispettare esattamente i 4.75 pollici
    plt.savefig(output_file, dpi=300, bbox_inches=None)
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import os


def plot_consensus_alignment_standardized(
    reference_seq: str,
    scaffolds: list,
    cdrs: dict,
    colors: dict,
    output_file: str = "../../../../figures/fig4e_consensus_alignment.svg",
    show_figure: bool = True
):

    def chars_equal(a, b):
        return (a in ['L', 'I'] and b in ['L', 'I']) or a == b

    def find_alignment_offset(ref, seq):
        for pos in range(len(ref) - len(seq) + 1):
            if all(chars_equal(ref[pos+i], seq[i]) for i in range(len(seq))):
                return pos
        return ref.find(seq)

    char_w = 0.12 
    calculated_width = (len(reference_seq) * char_w) + 2.0
    calculated_height = 1.5 + (len(scaffolds) * 0.5)

    fig, ax = plt.subplots(figsize=(calculated_width, calculated_height), layout='constrained')
    
    ax.axis('off')
    
    FONT_SIZE = 10 
    FONT_FAMILY = 'Arial'
    Y_REF = 0
    Y_STEP = -1.0
    
    
    ax.text(-2, Y_REF, "Reference", ha='right', va='center', fontsize=FONT_SIZE+1, fontweight='bold', color='#2c3e50')
    
    for i, char in enumerate(reference_seq):
        ax.text(i, Y_REF, char, ha='center', va='center', fontsize=FONT_SIZE, fontfamily=FONT_FAMILY)

    for cdr_name, (start, end) in cdrs.items():
        x0 = (start - 1) - 0.5
        width = (end - start + 1)
        
        rect_ref = patches.Rectangle(
            (x0, Y_REF - 0.3), width, 0.6, 
            facecolor=colors.get(cdr_name, 'gray'), alpha=0.3, zorder=0
        )
        ax.add_patch(rect_ref)
        
        ax.text(
            (start - 1 + end)/2 - 0.5, Y_REF + 0.5, cdr_name, 
            ha='center', va='bottom', fontsize=8, fontweight='bold', color=colors.get(cdr_name, 'black')
        )

    for idx, (scaf_name, scaf_seq) in enumerate(scaffolds):
        y_pos = Y_REF + ((idx + 1) * Y_STEP)
        
        offset = find_alignment_offset(reference_seq, scaf_seq)
        if offset == -1: continue
            
        ax.text(-2, y_pos, scaf_name, ha='right', va='center', fontsize=FONT_SIZE+1, color='#2c3e50')
        
        for i, char in enumerate(scaf_seq):
            x_pos = offset + i
            ax.text(x_pos, y_pos, char, ha='center', va='center', fontsize=FONT_SIZE, fontfamily=FONT_FAMILY)
            
        for cdr_name, (start, end) in cdrs.items():
            cdr_start_0, cdr_end_0 = start - 1, end - 1
            scaf_start, scaf_end = offset, offset + len(scaf_seq) - 1
            ov_start = max(cdr_start_0, scaf_start)
            ov_end = min(cdr_end_0, scaf_end)
            
            if ov_start <= ov_end:
                rect_scaf = patches.Rectangle(
                    (ov_start - 0.5, y_pos - 0.3), ov_end - ov_start + 1, 0.6, 
                    facecolor=colors.get(cdr_name, 'gray'), alpha=0.3, zorder=0
                )
                ax.add_patch(rect_scaf)

    y_ruler = Y_REF + ((len(scaffolds) + 0.5) * Y_STEP)
    for i in range(0, len(reference_seq), 10):
        label_num = i + 1
        ax.text(i, y_ruler, str(label_num), ha='center', va='top', fontsize=7, color='gray')

    ax.set_xlim(-15, len(reference_seq) + 2)
    ax.set_ylim(y_ruler - 1, Y_REF + 1.5)
    
    os.makedirs(os.path.dirname(output_file) if os.path.dirname(output_file) else '.', exist_ok=True)
    
    plt.savefig(output_file, dpi=300, bbox_inches=None)
    
    if show_figure:
        plt.show()

In [ ]:
plot_consensus_alignment_standardized()